# 08 — Cross-Project Evaluation — FINAL LOPO

Thí nghiệm chính của đề tài.

Mỗi fold:
1. hold out đúng 1 `repository::project`;
2. train trên tất cả project còn lại;
3. đánh giá Cox và RSF trên project chưa xuất hiện trong train.

Bản này:
- mặc định FINAL (`fast_mode: false`);
- không sample held-out test cho Harrell/IPCW/AUC;
- chỉ sample IBS để tránh MemoryError;
- chạy Day-0 và Day-7;
- lưu CSV sau mỗi model/fold;
- có resume nếu notebook bị dừng.

In [1]:
from pathlib import Path
import sys, yaml, json, time, gc, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

SEED = int(CFG["random_seed"])
MODES = list(CFG.get("evaluation", {}).get("modes", ["day0"]))

print("ROOT =", ROOT)
print("random_seed =", SEED)
print("modes =", MODES)

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42
modes = ['day0', 'day7']


In [2]:
import hashlib

from src.features import feature_spec, sample_training_rows
from src.models import fit_cox, fit_rsf
from src.evaluation import evaluate_survival_model
from src.splitting import add_project_uid

table_dir = ROOT / "results" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)

cp_cfg = CFG.get("cross_project", {})
FAST_MODE = bool(cp_cfg.get("fast_mode", False))
RESUME = bool(cp_cfg.get("resume", True))
FORCE_RERUN = bool(cp_cfg.get("force_rerun", False))
SMOKE_FOLDS = int(cp_cfg.get("smoke_folds", 3))

print("FAST_MODE =", FAST_MODE)
print("RESUME =", RESUME)
print("FORCE_RERUN =", FORCE_RERUN)

FAST_MODE = False
RESUME = True
FORCE_RERUN = False


In [3]:
def make_run_signature(mode: str) -> str:
    payload = {
        "mode": mode,
        "seed": SEED,
        "max_train_rows": CFG["models"].get("max_train_rows"),
        "cox": CFG["models"]["cox"],
        "rsf": CFG["models"]["rsf"],
        "evaluation": {
            "horizons_days": CFG["evaluation"]["horizons_days"],
            "max_ibs_rows": CFG["evaluation"]["max_ibs_rows"],
        },
    }
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:12]


def load_existing_results(path: Path, signature: str) -> pd.DataFrame:
    if FORCE_RERUN or not RESUME or not path.exists():
        return pd.DataFrame()

    old = pd.read_csv(path)
    if "run_signature" not in old.columns:
        return pd.DataFrame()

    return old[
        old["run_signature"].astype(str) == str(signature)
    ].copy()


def completed_keys(df: pd.DataFrame):
    if df.empty or "status" not in df.columns:
        return set()

    good = df[df["status"].astype(str) == "ok"]
    return set(zip(
        good["heldout_uid"].astype(str),
        good["model"].astype(str),
    ))


def append_and_save(current: pd.DataFrame, row: dict, path: Path) -> pd.DataFrame:
    updated = pd.concat([current, pd.DataFrame([row])], ignore_index=True)
    updated.to_csv(path, index=False)
    return updated

In [ ]:
all_mode_summaries = []

for mode in MODES:
    print("\n" + "#" * 110)
    print("CROSS-PROJECT MODE:", mode)
    print("#" * 110)

    data_path = ROOT / "data" / "processed" / f"model_{mode}.parquet"
    if not data_path.exists():
        raise FileNotFoundError(
            f"Không thấy {data_path}. Hãy chạy 04_feature_engineering.ipynb."
        )

    data = add_project_uid(pd.read_parquet(data_path))

    project_table = (
        data.groupby(["project_uid", "repository", "project"], dropna=False)
        .agg(
            issues=("issue_key", "size"),
            events=("event", "sum"),
            median_duration=("duration_days", "median"),
        )
        .reset_index()
        .sort_values("project_uid")
    )

    all_uids = project_table["project_uid"].astype(str).tolist()

    if FAST_MODE:
        heldout_uids = all_uids[:min(SMOKE_FOLDS, len(all_uids))]
        suffix = "SMOKE"
        print("SMOKE TEST ONLY:", heldout_uids)
    else:
        heldout_uids = all_uids
        suffix = "FINAL"
        print("FINAL LOPO folds:", len(heldout_uids))

    signature = make_run_signature(mode)
    out_path = table_dir / f"cross_project_{mode}_{suffix}.csv"

    results = load_existing_results(out_path, signature)
    done = completed_keys(results)

    print("Run signature:", signature)
    print("Already completed model-fold pairs:", len(done))

    strict = bool(CFG["features"]["strict_no_leakage"])
    landmark = int(CFG["features"]["landmark_days"])

    numeric_cols, categorical_cols = feature_spec(
        mode,
        strict_no_leakage=strict,
        landmark_days=landmark,
    )

    max_rows = 20000 if FAST_MODE else CFG["models"].get("max_train_rows")
    eval_cfg = CFG["evaluation"]

    for fold_i, heldout_uid in enumerate(heldout_uids, start=1):
        print(f"\n[{fold_i}/{len(heldout_uids)}] Hold out: {heldout_uid}")

        train = data[data["project_uid"].astype(str) != str(heldout_uid)].copy()
        test = data[data["project_uid"].astype(str) == str(heldout_uid)].copy()

        if test.empty:
            print("  WARNING: empty test set")
            continue

        heldout_repository = str(test["repository"].iloc[0])
        heldout_project = str(test["project"].iloc[0])

        train_fit = sample_training_rows(
            train,
            max_rows=max_rows,
            random_state=SEED + fold_i,
        )

        print("  train used:", len(train_fit), "/", len(train))
        print("  held-out test:", len(test))

        # ---------------- Cox ----------------
        cox_key = (str(heldout_uid), "CoxPH")
        if cox_key in done:
            print("  CoxPH: skip (already completed)")
        else:
            started = time.time()
            try:
                alpha = float(CFG["models"]["cox"]["alpha"])
                cox = fit_cox(
                    train_fit,
                    numeric_cols,
                    categorical_cols,
                    alpha=alpha,
                )

                if not np.all(np.isfinite(np.asarray(cox["model"].coef_))):
                    raise RuntimeError("Cox produced non-finite coefficients.")

                metrics = evaluate_survival_model(
                    cox,
                    train_df=train_fit,
                    test_df=test,
                    horizons_days=eval_cfg["horizons_days"],
                    max_ibs_rows=eval_cfg["max_ibs_rows"],
                    survival_batch_size=eval_cfg["survival_batch_size"],
                    random_state=SEED + fold_i,
                )

                row = {
                    "run_signature": signature,
                    "mode": mode,
                    "heldout_uid": str(heldout_uid),
                    "heldout_repository": heldout_repository,
                    "heldout_project": heldout_project,
                    "model": "CoxPH",
                    "status": "ok",
                    "error": "",
                    "train_rows": len(train_fit),
                    "test_rows": len(test),
                    "wall_seconds": time.time() - started,
                    **metrics,
                }
                print("  CoxPH:", {k: row.get(k) for k in [
                    "harrell_c", "ipcw_c", "mean_dynamic_auc", "ibs"
                ]})
            except Exception as e:
                row = {
                    "run_signature": signature,
                    "mode": mode,
                    "heldout_uid": str(heldout_uid),
                    "heldout_repository": heldout_repository,
                    "heldout_project": heldout_project,
                    "model": "CoxPH",
                    "status": "error",
                    "error": repr(e),
                    "train_rows": len(train_fit),
                    "test_rows": len(test),
                    "wall_seconds": time.time() - started,
                }
                print("  CoxPH ERROR:", repr(e))

            results = append_and_save(results, row, out_path)
            if "cox" in locals():
                del cox
            gc.collect()

        # ---------------- RSF ----------------
        rsf_key = (str(heldout_uid), "RSF")
        if rsf_key in done:
            print("  RSF: skip (already completed)")
        else:
            started = time.time()
            try:
                rsf_cfg = CFG["models"]["rsf"].copy()

                if FAST_MODE:
                    rsf_cfg["n_estimators"] = min(int(rsf_cfg["n_estimators"]), 20)
                    rsf_cfg["max_depth"] = 10
                    rsf_cfg["n_jobs"] = 1

                rsf_cfg["random_state"] = SEED + fold_i

                rsf = fit_rsf(
                    train_fit,
                    numeric_cols,
                    categorical_cols,
                    **rsf_cfg,
                )

                metrics = evaluate_survival_model(
                    rsf,
                    train_df=train_fit,
                    test_df=test,
                    horizons_days=eval_cfg["horizons_days"],
                    max_ibs_rows=eval_cfg["max_ibs_rows"],
                    survival_batch_size=eval_cfg["survival_batch_size"],
                    random_state=SEED + 1000 + fold_i,
                )

                row = {
                    "run_signature": signature,
                    "mode": mode,
                    "heldout_uid": str(heldout_uid),
                    "heldout_repository": heldout_repository,
                    "heldout_project": heldout_project,
                    "model": "RSF",
                    "status": "ok",
                    "error": "",
                    "train_rows": len(train_fit),
                    "test_rows": len(test),
                    "wall_seconds": time.time() - started,
                    **metrics,
                }
                print("  RSF:", {k: row.get(k) for k in [
                    "harrell_c", "ipcw_c", "mean_dynamic_auc", "ibs"
                ]})
            except Exception as e:
                row = {
                    "run_signature": signature,
                    "mode": mode,
                    "heldout_uid": str(heldout_uid),
                    "heldout_repository": heldout_repository,
                    "heldout_project": heldout_project,
                    "model": "RSF",
                    "status": "error",
                    "error": repr(e),
                    "train_rows": len(train_fit),
                    "test_rows": len(test),
                    "wall_seconds": time.time() - started,
                }
                print("  RSF ERROR:", repr(e))

            results = append_and_save(results, row, out_path)
            if "rsf" in locals():
                del rsf
            gc.collect()

        del train, test, train_fit
        gc.collect()

    # Macro summary: mỗi held-out project có trọng số bằng nhau
    final_results = pd.read_csv(out_path)
    final_results = final_results[
        final_results["run_signature"].astype(str) == str(signature)
    ].copy()
    ok = final_results[final_results["status"] == "ok"].copy()

    metric_cols = [
        c for c in [
            "harrell_c", "ipcw_c",
            "auc_30", "auc_60", "auc_90",
            "mean_dynamic_auc",
            "ibs", "km_ibs", "ibs_gain_vs_km",
        ]
        if c in ok.columns
    ]

    summary_rows = []
    for model_name, g in ok.groupby("model"):
        summary_row = {
            "mode": mode,
            "model": model_name,
            "successful_folds": int(g["heldout_uid"].nunique()),
        }

        for metric in metric_cols:
            vals = pd.to_numeric(g[metric], errors="coerce")
            summary_row[f"{metric}_mean"] = float(vals.mean())
            summary_row[f"{metric}_std"] = float(vals.std(ddof=1))
            summary_row[f"{metric}_median"] = float(vals.median())
            summary_row[f"{metric}_min"] = float(vals.min())
            summary_row[f"{metric}_max"] = float(vals.max())

        summary_rows.append(summary_row)

    macro = pd.DataFrame(summary_rows)
    macro.to_csv(
        table_dir / f"cross_project_{mode}_macro_{suffix}.csv",
        index=False,
    )
    all_mode_summaries.append(macro)

    print("\nMACRO SUMMARY")
    display(macro)

    del data
    gc.collect()

all_macro = pd.concat(all_mode_summaries, ignore_index=True)
suffix = "SMOKE" if FAST_MODE else "FINAL"

all_macro.to_csv(
    table_dir / f"cross_project_macro_ALL_MODES_{suffix}.csv",
    index=False,
)

all_macro


##############################################################################################################
CROSS-PROJECT MODE: day0
##############################################################################################################
FINAL LOPO folds: 12
Run signature: de2d050dd049
Already completed model-fold pairs: 0

[1/12] Hold out: Apache::FLEX
  train used: 50000 / 800253
  held-out test: 35390
  CoxPH: {'harrell_c': 0.534580252473745, 'ipcw_c': 0.5319708947344377, 'mean_dynamic_auc': 0.5685662386120224, 'ibs': 0.1848629297652716}
  RSF: {'harrell_c': 0.5232293588826633, 'ipcw_c': 0.5205294109880795, 'mean_dynamic_auc': 0.5534892028828476, 'ibs': 0.1735846264583997}

[2/12] Hold out: Apache::HIVE
  train used: 50000 / 809912
  held-out test: 25731
  CoxPH: {'harrell_c': 0.6093325971346898, 'ipcw_c': 0.6080250164409108, 'mean_dynamic_auc': 0.657690789290033, 'ibs': 0.1817304610865975}
  RSF: {'harrell_c': 0.6161749341796139, 'ipcw_c': 0.6149202798294907, 'mean_dynami

,mode,model,successful_folds,harrell_c_mean,harrell_c_std,harrell_c_median,harrell_c_min,harrell_c_max,ipcw_c_mean,ipcw_c_std,...,km_ibs_mean,km_ibs_std,km_ibs_median,km_ibs_min,km_ibs_max,ibs_gain_vs_km_mean,ibs_gain_vs_km_std,ibs_gain_vs_km_median,ibs_gain_vs_km_min,ibs_gain_vs_km_max
0,day0,CoxPH,12,0.550318,0.054923,0.543593,0.452251,0.651056,0.546470,0.050048,...,0.232610,0.072724,0.202282,0.168937,0.409715,0.020352,0.045135,0.007628,-0.044460,0.111572
1,day0,RSF,12,0.550930,0.056044,0.544608,0.470859,0.634243,0.546402,0.050484,...,0.234316,0.071388,0.207550,0.165398,0.406615,0.015847,0.027112,0.008535,-0.021929,0.087636



##############################################################################################################
CROSS-PROJECT MODE: day7
##############################################################################################################
FINAL LOPO folds: 12
Run signature: 3a611df8afc6
Already completed model-fold pairs: 0

[1/12] Hold out: Apache::FLEX
  train used: 50000 / 392565
  held-out test: 27464
  CoxPH: {'harrell_c': 0.4807389486706444, 'ipcw_c': 0.47953655352510305, 'mean_dynamic_auc': 0.5002252757081166, 'ibs': 0.2796825669670548}


## Dùng số nào trong báo cáo?

Bảng chính:
- macro mean ± SD giữa các held-out project;
- Day-0 vs Day-7;
- Cox vs RSF.

Bảng phụ:
- metric từng held-out project.

Không dùng pooled 2-project test C-index làm bằng chứng cross-project chính.